Version 1

In [2]:
# ============================================================
# CELL: Stay-Point Detection (CENTROID-BASED) - MỘT USER
# ============================================================

import os
import glob
import random
import pandas as pd
import folium
from pathlib import Path
from gps.features.stay_point import StayPointDetector, DistanceMode

# === CẤU HÌNH ===
DATA_DIR = Path("../data/Geolife Trajectories 1.3/Data")
OUTPUT_DIR = Path("04_staypoint_maps")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Các threshold combinations cần test
THRESHOLD_PAIRS = [
    (10, 100),
    (20, 150),
    (30, 200),
    (45, 200),
    (60, 300),
]

N_USERS = 10
random.seed(42)

# === BƯỚC 1: Get random users ===
all_user_ids = [d.name for d in DATA_DIR.iterdir() if d.is_dir()]
sample_users = random.sample(all_user_ids, min(N_USERS, len(all_user_ids)))
print(f"📊 Testing với {len(sample_users)} users: {sample_users}")

# === BƯỚC 2: Load trajectory của 1 user ===
def load_user_trajectory(user_id):
    user_dir = DATA_DIR / user_id / "Trajectory"
    if not user_dir.exists():
        return None
    
    all_data = []
    for plt_file in user_dir.glob("*.plt"):
        try:
            df = pd.read_csv(
                plt_file,
                skiprows=6,
                names=['lat', 'lon', 'reserved', 'altitude', 
                       'date_days', 'date_str', 'time_str'],
                header=None
            )
            df['timestamp'] = pd.to_datetime(df['date_str'] + ' ' + df['time_str'])
            df['source_file'] = plt_file.name
            all_data.append(df)
        except:
            continue
    
    if not all_data:
        return None
    
    combined_df = pd.concat(all_data, ignore_index=True)
    combined_df = combined_df.sort_values('timestamp').reset_index(drop=True)
    return combined_df

# === BƯỚC 3: Detect với CENTROID mode ===
def detect_all_thresholds(traj_df, threshold_pairs):
    results = {}
    for time_th, dist_th in threshold_pairs:
        # ✅ SỬA: Thêm distance_mode=CENTROID
        detector = StayPointDetector(
            time_threshold_minutes=time_th,
            distance_threshold_meters=dist_th,
            distance_mode=DistanceMode.CENTROID
        )
        stay_pts = detector.detect(traj_df)
        results[(time_th, dist_th)] = stay_pts
        print(f"  Threshold ({time_th}min, {dist_th}m): {len(stay_pts)} stay-points")
    return results

# === BƯỚC 4: Tạo map ===
def create_map_with_selector(user_id, traj_df, stay_results, threshold_pairs):
    center_lat = traj_df['lat'].mean()
    center_lon = traj_df['lon'].mean()
    m = folium.Map(location=[center_lat, center_lon], zoom_start=12)
    
    # Raw trajectory (hidden by default)
    traj_group = folium.FeatureGroup(name='Raw Trajectory', show=False)
    trajectory = traj_df[['lat', 'lon']].values.tolist()
    folium.PolyLine(trajectory, weight=1, color='gray', opacity=0.3).add_to(traj_group)
    traj_group.add_to(m)
    
    # Stay-point layers
    for time_th, dist_th in threshold_pairs:
        layer_name = f"Stay-Points ({time_th}min, {dist_th}m)"
        layer_group = folium.FeatureGroup(name=layer_name, show=True)
        
        for i, sp in enumerate(stay_results[(time_th, dist_th)]):
            folium.CircleMarker(
                location=[sp.lat, sp.lon],
                radius=8,
                color='red',
                fill=True,
                fillColor='red',
                fillOpacity=0.7,
                popup=f"<b>SP #{i+1}</b><br>Time: {time_th}min<br>Dist: {dist_th}m<br>Duration: {sp.duration_minutes:.1f}min"
            ).add_to(layer_group)
        
        layer_group.add_to(m)
    
    folium.LayerControl(collapsed=False).add_to(m)
    
    # Title
    title_html = f'''
    <div style="position: fixed; top: 10px; left: 50px; z-index: 1000; 
                background-color: white; padding: 10px; border-radius: 5px;
                border: 2px solid gray;">
        <h4>User: {user_id}</h4>
        <p style="font-size: 12px;">Records: {len(traj_df):,} | Files: {traj_df['source_file'].nunique()}</p>
        <p style="font-size: 11px; color: blue;">✅ CENTROID-based detection</p>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(title_html))
    
    return m

# === BƯỚC 5: Process all users ===
for user_id in sample_users:
    
    if user_id == "163":
        continue
    
    print(f"\n{'='*60}")
    print(f"🔍 Processing user: {user_id}")
    print('='*60)
    
    traj_df = load_user_trajectory(user_id)
    if traj_df is None:
        print(f"  ❌ No data")
        continue
    
    print(f"  ✅ {len(traj_df):,} records, {traj_df['source_file'].nunique()} files")
    
    print(f"\n  📍 Detecting stay-points (CENTROID mode)...")
    stay_results = detect_all_thresholds(traj_df, THRESHOLD_PAIRS)
    
    print(f"\n  🗺️ Creating map...")
    m = create_map_with_selector(user_id, traj_df, stay_results, THRESHOLD_PAIRS)
    
    output_path = OUTPUT_DIR / f"staypoint_user_{user_id}.html"
    m.save(str(output_path))
    print(f"\n  ✅ Saved: {output_path}")

print(f"\n{'='*60}")
print(f"🎉 Hoàn tất! Maps ở: {OUTPUT_DIR}")

📊 Testing với 10 users: ['163', '028', '006', '070', '062', '057', '035', '026', '173', '139']

🔍 Processing user: 028
  ✅ 123,828 records, 64 files

  📍 Detecting stay-points (CENTROID mode)...
  Threshold (10min, 100m): 87 stay-points
  Threshold (20min, 150m): 70 stay-points
  Threshold (30min, 200m): 61 stay-points
  Threshold (45min, 200m): 50 stay-points
  Threshold (60min, 300m): 46 stay-points

  🗺️ Creating map...

  ✅ Saved: 04_staypoint_maps\staypoint_user_028.html

🔍 Processing user: 006
  ✅ 31,830 records, 28 files

  📍 Detecting stay-points (CENTROID mode)...
  Threshold (10min, 100m): 42 stay-points
  Threshold (20min, 150m): 33 stay-points
  Threshold (30min, 200m): 25 stay-points
  Threshold (45min, 200m): 19 stay-points
  Threshold (60min, 300m): 26 stay-points

  🗺️ Creating map...

  ✅ Saved: 04_staypoint_maps\staypoint_user_006.html

🔍 Processing user: 070
  ✅ 35,311 records, 11 files

  📍 Detecting stay-points (CENTROID mode)...
  Threshold (10min, 100m): 31 stay-

In [4]:
# ============================================================
# CELL: Stay-Point Detection (CENTROID-BASED) - MỘT USER
# ============================================================

import os
import glob
import random
import pandas as pd
import folium
from pathlib import Path
from gps.features.stay_point import StayPointDetector, DistanceMode

# === CẤU HÌNH ===
DATA_DIR = Path("../data/Geolife Trajectories 1.3/Data")
OUTPUT_DIR = Path("04_staypoint_maps_radius")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Các threshold combinations cần test
THRESHOLD_PAIRS = [
    (10, 100),
    (20, 150),
    (30, 200),
    (45, 200),
    (60, 300),
]

N_USERS = 10
random.seed(42)

# === BƯỚC 1: Get random users ===
all_user_ids = [d.name for d in DATA_DIR.iterdir() if d.is_dir()]
sample_users = random.sample(all_user_ids, min(N_USERS, len(all_user_ids)))
print(f"📊 Testing với {len(sample_users)} users: {sample_users}")

# === BƯỚC 2: Load trajectory của 1 user ===
def load_user_trajectory(user_id):
    user_dir = DATA_DIR / user_id / "Trajectory"
    if not user_dir.exists():
        return None
    
    all_data = []
    for plt_file in user_dir.glob("*.plt"):
        try:
            df = pd.read_csv(
                plt_file,
                skiprows=6,
                names=['lat', 'lon', 'reserved', 'altitude', 
                       'date_days', 'date_str', 'time_str'],
                header=None
            )
            df['timestamp'] = pd.to_datetime(df['date_str'] + ' ' + df['time_str'])
            df['source_file'] = plt_file.name
            all_data.append(df)
        except:
            continue
    
    if not all_data:
        return None
    
    combined_df = pd.concat(all_data, ignore_index=True)
    combined_df = combined_df.sort_values('timestamp').reset_index(drop=True)
    return combined_df

# === BƯỚC 3: Detect với CENTROID mode ===
def detect_all_thresholds(traj_df, threshold_pairs):
    results = {}
    for time_th, dist_th in threshold_pairs:
        # ✅ SỬA: Thêm distance_mode=CENTROID
        detector = StayPointDetector(
            time_threshold_minutes=time_th,
            distance_threshold_meters=dist_th,
            distance_mode=DistanceMode.CENTROID
        )
        stay_pts = detector.detect(traj_df)
        results[(time_th, dist_th)] = stay_pts
        print(f"  Threshold ({time_th}min, {dist_th}m): {len(stay_pts)} stay-points")
    return results

# === BƯỚC 4': Tạo map với RADIUS + CLUSTERED POINTS ===
def create_map_with_clusters(user_id, traj_df, stay_results, threshold_pairs, sample_fraction=0.1):
    """Tạo map với:
    1. Circle thể hiện bán kính stay-region
    2. Điểm trajectory được gán màu theo cluster
    """
    center_lat = traj_df['lat'].mean()
    center_lon = traj_df['lon'].mean()
    m = folium.Map(location=[center_lat, center_lon], zoom_start=12)
    
    # Raw trajectory (ẩn mặc định - chỉ tham khảo)
    traj_group = folium.FeatureGroup(name='Raw Trajectory (all)', show=False)
    trajectory = traj_df[['lat', 'lon']].values.tolist()
    folium.PolyLine(trajectory, weight=1, color='gray', opacity=0.2).add_to(traj_group)
    traj_group.add_to(m)
    
    # Stay-point layers với cluster points
    for time_th, dist_th in threshold_pairs:
        layer_name = f"Stay-Points ({time_th}min, {dist_th}m)"
        layer_group = folium.FeatureGroup(name=layer_name, show=True)
        
        # Tạo mapping: trajectory point → stay-point index
        # Bằng cách gán mỗi trajectory point vào cluster gần nhất
        sp_list = stay_results[(time_th, dist_th)]
        
        # Gán colors cho từng stay-point
        colors = [
            '#e41a1c', '#377eb8', '#4daf4a', '#984ea3', 
            '#ff7f00', '#ffff33', '#a65628', '#f781bf',
            '#999999', '#66c2a5'
        ]
        
        # Tạo dict để track points thuộc cluster nào
        # Vì StayPointDetector không lưu indices, ta cần gán lại
        # Đơn giản: gán mỗi trajectory point vào SP gần nhất trong thời gian đó
        
        # --- THÊM MỚI 1: Circle thể hiện bán kính ---
        for i, sp in enumerate(sp_list):
            color = colors[i % len(colors)]
            
            # Circle với bán kính = distance_threshold
            folium.Circle(
                location=[sp.lat, sp.lon],
                radius=dist_th,  # meters
                color=color,
                fill=True,
                fillColor=color,
                fillOpacity=0.1,  # Mờ
                weight=2,
                opacity=0.5,
                popup=f"<b>SP #{i+1}</b><br>Radius: {dist_th}m"
            ).add_to(layer_group)
            
            # --- THÊM MỚI 2: Điểm centroid nổi bật ---
            folium.CircleMarker(
                location=[sp.lat, sp.lon],
                radius=10,
                color='black',
                weight=2,
                fill=True,
                fillColor=color,
                fillOpacity=1,
                popup=f"<b>SP #{i+1}</b><br>Duration: {sp.duration_minutes:.1f}min<br>Points: {sp.num_points}"
            ).add_to(layer_group)
            
            # Thêm text label
            folium.Marker(
                location=[sp.lat, sp.lon],
                icon=folium.DivIcon(
                    icon_size=(30, 30),
                    icon_anchor=(15, 15),
                    html=f'<div style="font-size: 10px; font-weight: bold; color: black; '
                         f'background-color: white; border: 1px solid {color}; '
                         f'border-radius: 50%; width: 20px; height: 20px; '
                         f'text-align: center; line-height: 20px;">{i+1}</div>'
                )
            ).add_to(layer_group)
        
        # --- THÊM MỚI 3: Gán trajectory points vào clusters ---
        # Tạo DataFrame từ trajectory
        traj_df_sorted = traj_df.sort_values('timestamp').reset_index(drop=True)
        
        # Với mỗi stay-point, tìm các trajectory points trong khoảng thời gian & không gian
        for i, sp in enumerate(sp_list):
            color = colors[i % len(colors)]
            
            # Lọc points trong khoảng thời gian của stay-point
            mask = (
                (traj_df_sorted['timestamp'] >= sp.arrival_time) &
                (traj_df_sorted['timestamp'] <= sp.departure_time)
            )
            
            # Trong số đó, giữ lại những points gần centroid trong threshold
            cluster_points = []
            for _, row in traj_df_sorted[mask].iterrows():
                from gps.utils.geo import haversine_meters
                dist = haversine_meters(sp.lat, sp.lon, row['lat'], row['lon'])
                if dist <= dist_th:
                    cluster_points.append({
                        'lat': row['lat'],
                        'lon': row['lon'],
                        'dist': dist
                    })
            
            # Vẽ các điểm này với màu riêng
            if cluster_points:
                cluster_group = folium.FeatureGroup(name=f'Cluster {i+1} Points', show=True)
                
                # Điểm trong cluster (màu đậm hơn = gần centroid)
                for pt in cluster_points:
                    # Alpha theo khoảng cách: gần = đậm, xa = nhạt
                    alpha = max(0.3, 1 - pt['dist'] / dist_th)
                    
                    folium.CircleMarker(
                        location=[pt['lat'], pt['lon']],
                        radius=4,
                        color='black',
                        weight=1,
                        fill=True,
                        fillColor=color,
                        fillOpacity=alpha,
                    ).add_to(cluster_group)
                
                cluster_group.add_to(layer_group)
        
        layer_group.add_to(m)
    
    folium.LayerControl(collapsed=False).add_to(m)
    
    # Legend
    legend_html = '''
    <div style="position: fixed; bottom: 50px; right: 50px; z-index: 1000; 
                background-color: white; padding: 15px; border-radius: 5px;
                border: 2px solid gray; font-family: Arial; font-size: 12px;">
        <h4 style="margin: 0 0 10px 0;">Legend</h4>
        <p><span style="color: red;">🔴</span> Centroid</p>
        <p><span style="color: rgba(0,0,255,0.2);">⭕</span> Stay Region (radius)</p>
        <p><span style="color: blue;">🔵</span> Trajectory Points</p>
        <p><span style="color: gray;">- - -</span> Raw Trajectory</p>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    
    # Title
    title_html = f'''
    <div style="position: fixed; top: 10px; left: 50px; z-index: 1000; 
                background-color: white; padding: 10px; border-radius: 5px;
                border: 2px solid gray;">
        <h4>User: {user_id}</h4>
        <p style="font-size: 12px;">Records: {len(traj_df):,} | 
            Stay-Points: {len(stay_results[(threshold_pairs[0][0], threshold_pairs[0][1])])}</p>
        <p style="font-size: 11px; color: green;">✅ Circles = stay region radius</p>
        <p style="font-size: 11px; color: blue;">🔵 Points = cluster members</p>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(title_html))
    
    return m

# === BƯỚC 5': Process all users ===
for user_id in sample_users:
    
    if user_id == "163":
        continue
    
    print(f"\n{'='*60}")
    print(f"🔍 Processing user: {user_id}")
    print('='*60)
    
    traj_df = load_user_trajectory(user_id)
    if traj_df is None:
        print(f"  ❌ No data")
        continue
    
    print(f"  ✅ {len(traj_df):,} records, {traj_df['source_file'].nunique()} files")
    
    print(f"\n  📍 Detecting stay-points (CENTROID mode)...")
    stay_results = detect_all_thresholds(traj_df, THRESHOLD_PAIRS)
    
    print(f"\n  🗺️ Creating map with clusters...")
    m = create_map_with_clusters(user_id, traj_df, stay_results, THRESHOLD_PAIRS)
    
    output_path = OUTPUT_DIR / f"staypoint_cluster_user_{user_id}.html"
    m.save(str(output_path))
    print(f"  ✅ Saved: {output_path}")

print(f"\n{'='*60}")
print(f"🎉 Hoàn tất! Maps ở: {OUTPUT_DIR}")

📊 Testing với 10 users: ['163', '028', '006', '070', '062', '057', '035', '026', '173', '139']

🔍 Processing user: 028
  ✅ 123,828 records, 64 files

  📍 Detecting stay-points (CENTROID mode)...
  Threshold (10min, 100m): 87 stay-points
  Threshold (20min, 150m): 70 stay-points
  Threshold (30min, 200m): 61 stay-points
  Threshold (45min, 200m): 50 stay-points
  Threshold (60min, 300m): 46 stay-points

  🗺️ Creating map with clusters...
  ✅ Saved: 04_staypoint_maps\staypoint_cluster_user_028.html

🔍 Processing user: 006
  ✅ 31,830 records, 28 files

  📍 Detecting stay-points (CENTROID mode)...
  Threshold (10min, 100m): 42 stay-points
  Threshold (20min, 150m): 33 stay-points
  Threshold (30min, 200m): 25 stay-points
  Threshold (45min, 200m): 19 stay-points
  Threshold (60min, 300m): 26 stay-points

  🗺️ Creating map with clusters...
  ✅ Saved: 04_staypoint_maps\staypoint_cluster_user_006.html

🔍 Processing user: 070
  ✅ 35,311 records, 11 files

  📍 Detecting stay-points (CENTROID mo